In [0]:
%sql

--Objective: fetch the total sales amount across all salespeople 
--Concept to practice: flatten() >> turns nested arrays into one array

--Create the array in a temp view
CREATE OR REPLACE TEMP VIEW sales AS
SELECT * FROM VALUES
(1, ARRAY(100,200)),
(2, ARRAY(50, 75)),
(3, ARRAY(300, 25)),
(4, ARRAY(500, 50))
AS sales(id, performance);

SELECT * FROM sales;

--Creates a nested array
CREATE OR REPLACE TEMP VIEW sales AS
SELECT * FROM VALUES
  (1, named_struct('sales', ARRAY(100, 200))),
  (2, named_struct('sales', ARRAY(50, 75))),
  (3, named_struct('sales', ARRAY(300, 25))),
(4, named_struct('sales', ARRAY(500, 50)))
AS sales(id, performance);


SELECT * FROM sales;


--collect_list
--collect_list is best used for row-to-array transformations within a Spark column
--it does not eliminate duplicates
--collect_list is an aggregate function, which is why the GROUP BY is added
SELECT
id,
collect_list(performance) as performance
FROM sales
GROUP BY id;

--flatten--
--flatten transforms an array of arrays into a single array
SELECT
    flatten(collect_list(performance.sales)) AS flattened_sales
FROM sales;
 



In [0]:
%sql
--creating this second nested array to see duplicates get eliminated by collect_set
CREATE OR REPLACE TEMP VIEW basket AS
SELECT * FROM VALUES
  (1, named_struct('basket', ARRAY("apple", "bread", "coffee", "milk"))),
  (2, named_struct('basket', ARRAY("apple", "apple", "banana", "banana"))),
  (3, named_struct('basket', ARRAY("bread", "eggs", "banana"))),
(4, named_struct('basket', ARRAY("bread", "eggs", "banana", "banana")))
AS basket(customer_id, items);

SELECT * FROM basket;

--collect_set
--collect_set is best used for eliminating duplicates while grouping rows into an array
--collect_set is an aggregate function, which is why the GROUP BY is added
--Objective: return a list of items in customer's basket by name, eliminating duplicates
SELECT
customer_id,
collect_set(items.basket) as items
from basket
GROUP BY customer_id;
--why did collect_set appear to not remove duplicates?
--because we established the array as a nested array, the collect_set function acknowledges each nested array as its own value 
--the elements inside items are not being treated individually

CREATE OR REPLACE TEMP VIEW basket AS
SELECT * FROM VALUES
  (1, ARRAY("apple", "bread", "coffee", "milk")),
  (2, ARRAY("apple", "apple", "banana", "banana")),
  (3, ARRAY("bread", "eggs", "banana")),
(4,  ARRAY("bread", "eggs", "banana", "banana"))
AS basket(customer_id, items);


SELECT
customer_id,
collect_set(items) as items
from basket
GROUP BY customer_id;
--this is still returning multiple bananas in each order 
--the collect_set function is treating the entire array as a single value/element


--Let's use explode to fix this
--explode()--
--explode() takes each element of an array and "explodes" it to become a row 
--with each item on its own row, deduplication from collect_set() will be able to work properly
SELECT
customer_id,
explode(items) as item
from basket;



SELECT
    customer_id,
    collect_set(item) AS items
FROM (
    SELECT
        customer_id,
        explode(items) AS item
    FROM basket
)
GROUP BY customer_id;
--Success!
--1	["apple","bread","milk","coffee"]
--3	["eggs","banana","bread"]
--2	["apple","banana"]
--4	["banana","bread","eggs"]


SELECT
flatten(collect_set(items)) as item
FROM basket;
--flatten is transformational, not aggregate, no GROUP BY is needed 
--one flattened list, but still working off of the nested array so we get those duplicates
--["apple","bread","coffee","milk","apple","apple","banana","banana","bread","eggs","banana","bread","eggs","banana","banana"]
--to remove duplicates, we will explode the array first, then collect_set to remove the duplicates

SELECT
    collect_set(item) AS items
FROM (
    SELECT
        explode(items) AS item
    FROM basket
);
--explode >> turns every element (every array in the nested structure) into its own row to be acknowledged by the next functions
--collect_set reads through rows with each row being its own entity, and removes duplicates to create one full "set" to present
--voila! a deduplicated list 
--["bread","milk","apple","coffee","banana","eggs"]
